# Mount Google Drive (only for checkpoints, dataset comes from GitHub)
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repository (includes dataset in data/hit-uav/)
import os
if os.path.exists('SGGF-Net'):
    print('Repository already exists, pulling latest changes...')
    %cd SGGF-Net
    !git pull origin main
else:
    !git clone https://github.com/HarishSankarK/SGGF-Net.git
    %cd SGGF-Net

# Verify dataset is included
!ls -la data/hit-uav/ 2>/dev/null && echo "✓ Dataset found in repository!" || echo "⚠ Dataset not found"


In [ ]:
# Verify we're in the right directory
import os
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")


In [ ]:
# Setup TPU Runtime and Install PyTorch XLA
# Make sure TPU runtime is enabled: Runtime → Change runtime type → TPU

import os
print("Checking TPU runtime...")
print(f"COLAB_TPU_ADDR: {os.environ.get('COLAB_TPU_ADDR', 'Not set (TPU runtime not enabled)')}")

# Install PyTorch XLA for TPU support
print("\nInstalling PyTorch XLA for TPU...")
!pip install torch torchvision
!pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

# Verify TPU setup
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    
    # Use new API (torch_xla 2.9+)
    try:
        device = torch_xla.device()
        print(f'\n✓ TPU available: {device}')
        is_tpu_available = True
        
        # Try to get world size
        try:
            world_size = xm.xla_world_size()
            print(f'✓ TPU cores: {world_size}')
        except AttributeError:
            try:
                world_size = torch_xla._XLAC._xla_get_replication_devices_count()
                print(f'✓ TPU cores: {world_size}')
            except:
                print('✓ TPU device initialized (core count unavailable)')
    except (RuntimeError, Exception) as e:
        # Fallback to old API
        try:
            device = xm.xla_device()
            print(f'\n✓ TPU available (old API): {device}')
            is_tpu_available = True
        except (RuntimeError, Exception) as e2:
            print(f'\n⚠ TPU initialization failed: {e2}')
            print('⚠ Make sure TPU runtime is enabled: Runtime → Change runtime type → TPU')
            print('⚠ Then: Runtime → Restart runtime')
            is_tpu_available = False
except RuntimeError as e:
    print(f'\n⚠ TPU initialization failed: {e}')
    print('⚠ Make sure TPU runtime is enabled: Runtime → Change runtime type → TPU')
    print('⚠ Then: Runtime → Restart runtime')
    is_tpu_available = False
except Exception as e:
    print(f'\n⚠ Error setting up TPU: {e}')
    is_tpu_available = False

if is_tpu_available:
    print('\n✅ TPU setup complete! You can now use train_tpu.py')
else:
    print('\n❌ TPU not available. Use GPU training (Option A) instead.')


In [ ]:
# Dataset is already in the repository!
# Just verify it's there
import os

if os.path.exists('data/hit-uav'):
    print("✓ Dataset found in data/hit-uav/")
    print(f"  Images: {len([f for f in os.listdir('data/hit-uav/images/train') if f.endswith('.jpg')])} training images")
    print(f"  Labels: {len([f for f in os.listdir('data/hit-uav/labels/train') if f.endswith('.txt')])} training labels")
    print("\nDataset structure:")
    !ls -la data/hit-uav/
else:
    print("⚠ Dataset not found. Make sure you've pushed it to GitHub.")


## Step 5: Start Training


In [ ]:
## Step 5: Start Training

# Setup checkpoint directory in Google Drive
import os
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f'Checkpoints will be saved to: {drive_checkpoint_dir}')

# Train the model on GPU (checkpoints saved to Drive)
# GPU-OPTIMIZED: All optimizations applied
# - Learning rate: 0.0005 (prevents NaN)
# - Batch size: 1 (GPU memory optimized)
# - Max size: 800 (balanced for GPU memory)
# - Warmup + Cosine LR: faster convergence
# - GFEM optimized: patch_size=32, embed_dim=192, num_heads=6
# - All NaN fixes and speed optimizations applied
# NOTE: If CUDA is not available, the script will automatically use CPU
import torch
if torch.cuda.is_available():
    print(f"✓ Training on GPU: {torch.cuda.get_device_name(0)}")
    device_arg = "cuda"
else:
    print("⚠ CUDA not available, training on CPU (will be slower)")
    print("⚠ Make sure you ran Step 2 to install CUDA PyTorch")
    device_arg = "cpu"

!python scripts/train.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 1 \
    --grad_accum_steps 1 \
    --num_epochs 50 \
    --lr 0.0005 \
    --max_size 800 \
    --warmup_epochs 5 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --device {device_arg} \
    --val_freq 10 \
    --use_amp


## Step 6: Evaluate Model


### Option B: TPU Training Command

**Requirements:**
- TPU runtime enabled (Runtime → Change runtime type → TPU)
- Step 2B completed successfully
- TPU initialization successful


In [ ]:
# Train the model on TPU v5e-1 (checkpoints saved to Drive)
# TPU v5e-1 OPTIMIZED: All optimizations applied for TPU v5e-1
# - Learning rate: 0.0005 (prevents NaN)
# - Batch size: 16 (TPU v5e-1 can handle batch_size=16-32 efficiently)
# - Max size: 800 (balanced for TPU memory)
# - Warmup + Cosine LR: faster convergence
# - GFEM optimized: patch_size=32, embed_dim=192, num_heads=6
# - TPU-specific: Uses torch_xla, parallel data loading, TPU mark_step
# TPU v5e-1 has 8 cores, 16GB HBM per core, and can train 3-5x faster than GPU!
# 
# NOTE: If TPU initialization fails, the script will automatically fall back to GPU/CPU
!python scripts/train_tpu.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 16 \
    --grad_accum_steps 1 \
    --num_epochs 50 \
    --lr 0.0005 \
    --max_size 800 \
    --warmup_epochs 5 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --val_freq 10 \
    --use_amp


In [ ]:
# Evaluate on test set (using checkpoint from Drive)
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

!python scripts/evaluate.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --checkpoint {drive_checkpoint_dir}/best.pth \
    --num_classes 6 \
    --split test


## Step 7: Resume Training from Drive Checkpoint


In [ ]:
# Resume training from a checkpoint saved in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
import os
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'

if os.path.exists(latest_checkpoint):
    resume_from = latest_checkpoint
    print(f'Resuming from: {resume_from}')
elif os.path.exists(best_checkpoint):
    resume_from = best_checkpoint
    print(f'Resuming from: {resume_from}')
else:
    resume_from = None
    print('No checkpoint found, starting fresh training')

# Resume training on GPU (using GPU-optimized settings)
# All optimizations applied: LR=0.0005, max_size=800, patch_size=32, batch_size=1 for GPU
if resume_from:
    !python scripts/train.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 1 \
        --grad_accum_steps 1 \
        --num_epochs 50 \
        --lr 0.0005 \
        --max_size 800 \
        --warmup_epochs 5 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --device cuda \
        --val_freq 10 \
        --use_amp
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


## Checkpoint Management

Checkpoints are automatically saved to Google Drive at:
`/content/drive/MyDrive/SGGF-Net-checkpoints/`

- `latest.pth` - Latest checkpoint (every epoch)
- `best.pth` - Best model based on mAP

These persist even after Colab session ends!


### Option B: Resume TPU Training


In [ ]:
# Resume training on TPU v5e-1 (using TPU-optimized settings)
# All optimizations applied: LR=0.0005, max_size=800, patch_size=32, batch_size=16 for TPU v5e-1
if resume_from:
    !python scripts/train_tpu.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 16 \
        --grad_accum_steps 1 \
        --num_epochs 50 \
        --lr 0.0005 \
        --max_size 800 \
        --warmup_epochs 5 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --val_freq 10 \
        --use_amp
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


In [ ]:
# List checkpoints in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
import os

if os.path.exists(drive_checkpoint_dir):
    print(f"Checkpoints in Drive ({drive_checkpoint_dir}):")
    checkpoints = os.listdir(drive_checkpoint_dir)
    for ckpt in checkpoints:
        if ckpt.endswith('.pth'):
            size = os.path.getsize(f'{drive_checkpoint_dir}/{ckpt}') / (1024*1024)  # MB
            print(f"  - {ckpt} ({size:.2f} MB)")
else:
    print(f"Checkpoint directory not found: {drive_checkpoint_dir}")
    print("Run Step 5 to start training and create checkpoints.")
